# 05b — LoRA: Phi-4-mini sweep + best-config transfer

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**This notebook (05b):** runs the supervisor's full Phase 5 recipe for Phi-4-mini:
1. **Sweep on BBBP** — rank ∈ {8, 16} × lr ∈ {1e-4, 5e-4} = **4 runs**
2. **Pick best config** by ROC-AUC
3. **Apply best config to BACE and ESOL** (no extra hyperparameter search)

**Total: 6 LoRA runs.** Adds 6 rows to the LoRA scoreboard.

**Self-contained:** re-implements the 05a harness so this notebook survives a fresh Colab VM. Long-form comments live in 05a; here we focus on the sweep.

**Estimated runtime on T4:** ~60–90 min total. Phi-4-mini fp16 (~7.6 GB weights) plus LoRA training overhead fits T4 (16 GB) **with care**: batch size 2, gradient checkpointing on, max length 256. If we OOM at fp16, the fallback is 4-bit QLoRA — documented in section 8.

**Note on access:** Phi-4-mini-instruct is sometimes gated. If section 8 returns 401, run the optional HF-login cell first.

## 1. Install + imports

In [ ]:
# Upgrade torchao first (Colab ships an old version PEFT rejects), then the rest.
%pip install -q --upgrade torchao
%pip install -q --upgrade transformers peft accelerate bitsandbytes rdkit pandas scikit-learn

In [ ]:
import os, random, gc, time, json
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, mean_absolute_error
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")

## 2. (Optional) Hugging Face login

Only needed if Phi-4-mini is gated on your HF account. If the model load in section 8 returns 401, uncomment and run this cell first.

In [ ]:
# from huggingface_hub import login
# login()  # paste HF token (read scope) when prompted

## 3. Mount Drive + paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR     = "/content/drive/MyDrive/chemistry-peft-fyp/data"
RESULTS_DIR  = "/content/drive/MyDrive/chemistry-peft-fyp/results"
ADAPTERS_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/adapters"
RESULTS_CSV  = f"{RESULTS_DIR}/lora_results.csv"
os.makedirs(RESULTS_DIR,  exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
print("Adapters dir:", ADAPTERS_DIR)
print("Results CSV :", RESULTS_CSV)

## 4. Load splits + dataset configs + helpers (same as 05a)

In [ ]:
def load_splits(name):
    p = name.lower()
    return (
        pd.read_csv(f"{DATA_DIR}/{p}_train.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_val.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_test.csv"),
    )

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}
for n, (tr, va, te) in splits.items():
    print(f"{n:5s}: train={len(tr)}  val={len(va)}  test={len(te)}")

ESOL_BUCKETS = [
    ("very_low",  -np.inf, -6.0, -7.5),
    ("low",       -6.0,    -4.0, -5.0),
    ("medium",    -4.0,    -2.0, -3.0),
    ("high",      -2.0,     0.0, -1.0),
    ("very_high",  0.0,    np.inf, 1.0),
]
ESOL_LABEL_WORDS = [b[0] for b in ESOL_BUCKETS]
ESOL_MIDPOINTS   = np.array([b[3] for b in ESOL_BUCKETS], dtype=np.float32)

def esol_continuous_to_bucket(val):
    for label, lo, hi, _ in ESOL_BUCKETS:
        if lo < val <= hi:
            return label
    return ESOL_BUCKETS[0][0]

@dataclass
class DatasetConfig:
    name: str
    task: str
    question: str
    label_words: List[str]
    positive_word: Optional[str]

DATASETS = {
    "BBBP": DatasetConfig("BBBP", "classification",
        "Does this molecule cross the blood-brain barrier?",
        ["yes", "no"], "yes"),
    "BACE": DatasetConfig("BACE", "classification",
        "Does this molecule inhibit the BACE-1 enzyme?",
        ["yes", "no"], "yes"),
    "ESOL": DatasetConfig("ESOL", "regression",
        "What is the aqueous solubility of this molecule?",
        ESOL_LABEL_WORDS, None),
}

def label_for_prompt(cfg, raw_label):
    if cfg.task == "classification":
        return cfg.positive_word if int(raw_label) == 1 else "no"
    return esol_continuous_to_bucket(float(raw_label))

def build_prompt(cfg, test_smiles):
    return f"{cfg.question}\nSMILES: {test_smiles}\nAnswer:"

IGNORE_INDEX = -100
MAX_LEN = 256

class PromptAnswerDataset(TorchDataset):
    def __init__(self, df, cfg, tokenizer):
        self.cfg = cfg
        self.tok = tokenizer
        self.rows = []
        for _, row in df.iterrows():
            prompt = build_prompt(cfg, row["smiles"])
            answer = label_for_prompt(cfg, row["label"])
            self.rows.append((prompt, answer, row["label"]))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        prompt, answer, _ = self.rows[i]
        prompt_ids = self.tok(prompt, add_special_tokens=False).input_ids
        answer_ids = self.tok(" " + answer, add_special_tokens=False).input_ids
        eos = [self.tok.eos_token_id] if self.tok.eos_token_id is not None else []
        input_ids = (prompt_ids + answer_ids + eos)[:MAX_LEN]
        labels    = ([IGNORE_INDEX] * len(prompt_ids) + answer_ids + eos)[:MAX_LEN]
        attention = [1] * len(input_ids)
        return {
            "input_ids":      torch.tensor(input_ids,  dtype=torch.long),
            "labels":         torch.tensor(labels,     dtype=torch.long),
            "attention_mask": torch.tensor(attention,  dtype=torch.long),
        }

def collate_pad(batch, pad_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    def pad_to(t, val):
        return torch.cat([t, torch.full((max_len - len(t),), val, dtype=t.dtype)])
    return {
        "input_ids":      torch.stack([pad_to(b["input_ids"],      pad_id) for b in batch]),
        "labels":         torch.stack([pad_to(b["labels"],         IGNORE_INDEX) for b in batch]),
        "attention_mask": torch.stack([pad_to(b["attention_mask"], 0) for b in batch]),
    }

print("Helpers ready.")

## 5. LoRA target modules + harness (Phi-4 specific)

In [ ]:
MODEL_ID = "microsoft/Phi-4-mini-instruct"
# Phi-4-mini uses a fused QKV linear named qkv_proj; output projection is o_proj.
# These are the standard targets for Llama-style attention; PEFT will match by leaf name.
TARGET_MODULES = ["qkv_proj", "o_proj"]

In [ ]:
@torch.no_grad()
def score_labels(model, tokenizer, prompt, label_words):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    out = model(**enc)
    next_logits = out.logits[0, -1, :]
    log_probs = torch.log_softmax(next_logits, dim=-1)
    token_ids_per_label = []
    for w in label_words:
        cands = set()
        for variant in (" " + w, w):
            t = tokenizer(variant, add_special_tokens=False).input_ids
            if t: cands.add(t[0])
        token_ids_per_label.append(sorted(cands))
    label_logps = [max(log_probs[i].item() for i in ids) for ids in token_ids_per_label]
    logps = np.array(label_logps, dtype=np.float64)
    logps -= logps.max()
    probs = np.exp(logps); probs /= probs.sum()
    return probs

def eval_model(model, tokenizer, dataset_name):
    cfg = DATASETS[dataset_name]
    _, _, te_df = splits[dataset_name]
    probs_all, y_true = [], []
    model.eval()
    for _, row in te_df.iterrows():
        probs_all.append(score_labels(model, tokenizer, build_prompt(cfg, row["smiles"]), cfg.label_words))
        y_true.append(row["label"])
    probs_all = np.stack(probs_all); y_true = np.array(y_true)
    if cfg.task == "classification":
        pos_idx = cfg.label_words.index(cfg.positive_word)
        p_pos = probs_all[:, pos_idx]
        pred  = (p_pos >= 0.5).astype(int)
        return {
            "primary_metric": "roc_auc",
            "primary_value":  float(roc_auc_score(y_true, p_pos)),
            "secondary_metric": "f1",
            "secondary_value":  float(f1_score(y_true, pred)),
        }
    else:
        preds_cont = (probs_all * ESOL_MIDPOINTS[None, :]).sum(axis=1)
        y_cont = y_true.astype(np.float32)
        return {
            "primary_metric": "rmse",
            "primary_value":  float(np.sqrt(mean_squared_error(y_cont, preds_cont))),
            "secondary_metric": "mae",
            "secondary_value":  float(mean_absolute_error(y_cont, preds_cont)),
        }

## 6. Phi-4 LoRA training function

Specialised for Phi-4-mini fp16 with **gradient checkpointing** and **batch size 2**. These two tweaks are what make a 3.8B model fp16 LoRA training fit on a 16 GB T4.

Memory model (rough):
- Weights: ~7.6 GB (fp16)
- Activations w/ grad checkpointing + bs=2 + seq=256: ~3–4 GB
- LoRA adapter + optimizer states: ~0.2 GB
- **Total ~12–13 GB**, leaving 3–4 GB headroom on T4.

If we OOM, the fallback is to drop `torch_dtype` to `bnb_4bit` (true QLoRA) — but that's Phase 6's job, so we'd note it and continue.

In [ ]:
def phi4_lora_train_eval(
    dataset_name,
    rank, lora_alpha=16, lora_dropout=0.05,
    lr=5e-4, epochs=3, batch_size=2,
    save_adapter_to=None,
):
    cfg = DATASETS[dataset_name]
    tr_df, _, _ = splits[dataset_name]

    print(f"\nLoading {MODEL_ID} (fp16)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    )

    # Gradient checkpointing trades compute for memory — essential for fp16 Phi-4 on T4
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    lora_cfg = LoraConfig(
        r=rank, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
        target_modules=TARGET_MODULES, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)

    trainable, total = 0, 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad: trainable += p.numel()
    print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.4f}%)")

    train_ds = PromptAnswerDataset(tr_df, cfg, tokenizer)
    pad_id = tokenizer.pad_token_id
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: collate_pad(b, pad_id),
    )

    optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    total_steps = len(train_loader) * epochs
    sched = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps,
    )

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            optim.step(); sched.step(); optim.zero_grad()
            epoch_loss += loss.item()
            if (step + 1) % 100 == 0:
                elapsed = time.time() - t0
                print(f"  epoch {epoch+1} step {step+1}/{len(train_loader)}  loss={loss.item():.4f}  ({elapsed:.0f}s)")
        print(f"  epoch {epoch+1}: mean_loss={epoch_loss/len(train_loader):.4f}")
    train_time = time.time() - t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == "cuda" else 0

    print("Evaluating on test set...")
    metrics = eval_model(model, tokenizer, dataset_name)

    if save_adapter_to:
        os.makedirs(save_adapter_to, exist_ok=True)
        model.save_pretrained(save_adapter_to)
        tokenizer.save_pretrained(save_adapter_to)
        print(f"Adapter saved → {save_adapter_to}")

    row = {
        "model":            MODEL_ID,
        "dataset":          dataset_name,
        "task":             cfg.task,
        "lora_rank":        rank,
        "lora_alpha":       lora_alpha,
        "lora_lr":          lr,
        "epochs":           epochs,
        "trainable_params": trainable,
        "total_params":     total,
        "trainable_pct":    round(100*trainable/total, 4),
        "peak_mem_mb":      round(peak_mem_mb, 1),
        "train_time_s":     round(train_time, 1),
        "metric_primary":   metrics["primary_metric"],
        "value_primary":    round(metrics["primary_value"],   4),
        "metric_secondary": metrics["secondary_metric"],
        "value_secondary":  round(metrics["secondary_value"], 4),
    }

    del model, optim, sched, train_loader, train_ds
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return row

print("Phi-4 LoRA harness ready.")

## 7. Scoreboard append helper

In [ ]:
LORA_COLS = [
    "model", "dataset", "task",
    "lora_rank", "lora_alpha", "lora_lr", "epochs",
    "trainable_params", "total_params", "trainable_pct",
    "peak_mem_mb", "train_time_s",
    "metric_primary", "value_primary",
    "metric_secondary", "value_secondary",
]

def append_lora_row(row):
    new_df = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        existing = pd.read_csv(RESULTS_CSV)
        key = ["model", "dataset", "lora_rank", "lora_lr"]
        existing = existing[~existing[key].apply(tuple, axis=1).isin(
            new_df[key].apply(tuple, axis=1)
        )]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined[LORA_COLS]
    combined.to_csv(RESULTS_CSV, index=False)
    print(f"Wrote 1 row. LoRA scoreboard now: {len(combined)} rows.")
    return combined

## 8. The BBBP sweep — 4 runs

Grid: rank ∈ {8, 16} × lr ∈ {1e-4, 5e-4}. Each run trains from a fresh base model (no warm-starting across runs — that would be cheating on the sweep).

Each adapter saves to its own folder so we can reload the winner later. Folder name encodes the hyperparameters for traceability.

In [ ]:
SWEEP_GRID = [
    {"rank": 8,  "lr": 1e-4},
    {"rank": 8,  "lr": 5e-4},
    {"rank": 16, "lr": 1e-4},
    {"rank": 16, "lr": 5e-4},
]

bbbp_sweep_rows = []
for i, hp in enumerate(SWEEP_GRID, 1):
    adapter_dir = f"{ADAPTERS_DIR}/phi4_bbbp_r{hp['rank']}_lr{hp['lr']:.0e}"
    print(f"\n{'='*60}\nSweep run {i}/4 — rank={hp['rank']}, lr={hp['lr']:.0e}\n{'='*60}")
    row = phi4_lora_train_eval(
        dataset_name="BBBP",
        rank=hp["rank"], lr=hp["lr"],
        epochs=3, batch_size=2,
        save_adapter_to=adapter_dir,
    )
    bbbp_sweep_rows.append(row)
    append_lora_row(row)

print("\n--- BBBP sweep summary ---")
print(pd.DataFrame(bbbp_sweep_rows)[["lora_rank","lora_lr","value_primary","value_secondary","peak_mem_mb","train_time_s"]].to_string(index=False))

## 9. Pick the best BBBP config

In [ ]:
sweep_df = pd.DataFrame(bbbp_sweep_rows)
best_idx = sweep_df["value_primary"].idxmax()   # highest AUC
best = sweep_df.loc[best_idx]
print("Best BBBP config:")
print(f"  rank   = {int(best['lora_rank'])}")
print(f"  lr     = {best['lora_lr']}")
print(f"  AUC    = {best['value_primary']}")
print(f"  F1     = {best['value_secondary']}")

BEST_RANK = int(best["lora_rank"])
BEST_LR   = float(best["lora_lr"])

## 10. Transfer best config to BACE + ESOL

No further hyperparameter search per the supervisor's spec — same rank + lr applied to the other two datasets.

In [ ]:
transfer_rows = []
for ds_name in ["BACE", "ESOL"]:
    adapter_dir = f"{ADAPTERS_DIR}/phi4_{ds_name.lower()}_r{BEST_RANK}_lr{BEST_LR:.0e}"
    print(f"\n{'='*60}\nTransfer → {ds_name}  (rank={BEST_RANK}, lr={BEST_LR:.0e})\n{'='*60}")
    row = phi4_lora_train_eval(
        dataset_name=ds_name,
        rank=BEST_RANK, lr=BEST_LR,
        epochs=3, batch_size=2,
        save_adapter_to=adapter_dir,
    )
    transfer_rows.append(row)
    append_lora_row(row)

print("\n--- Transfer summary ---")
print(pd.DataFrame(transfer_rows)[["dataset","value_primary","value_secondary","peak_mem_mb","train_time_s"]].to_string(index=False))

## 11. Final Phi-4 LoRA view

In [ ]:
phi4_all = pd.read_csv(RESULTS_CSV)
phi4_all = phi4_all[phi4_all["model"] == MODEL_ID]
print(f"Phi-4-mini LoRA rows: {len(phi4_all)}\n")
print(phi4_all.to_string(index=False))

## 12. Done

**Phase 5 — Phi-4-mini half complete:**
- ✅ 4-run BBBP sweep (rank × lr grid)
- ✅ Best config picked by AUC
- ✅ Best config applied to BACE + ESOL
- ✅ 6 LoRA results rows written, 6 adapters saved to Drive

**Compare to Phase 4 (rough sanity):**
```
Phi-4 prompting retrieval BBBP:  AUC 0.678
Phi-4 prompting retrieval BACE:  AUC 0.812
Phi-4 prompting retrieval ESOL:  RMSE 1.941
```
Any LoRA row that beats these would be a strong Phase 5 result. Even rough parity at <0.5% of parameters is a story.

**Next: 05c — SmolLM3-3B + GPT-2 LoRA grids.** Same recipe (4-run BBBP sweep + transfer × 2 models) = 12 more rows. Total Phase 5 contribution will be 18 LoRA rows.